In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
import sqlite3
import time

In [19]:
#importing ticker spot gold, eurusd, gbpusd, btcusd and NQ futures values
tickers = ['GC=F','EURUSD=X','GBPUSD=X','BTC-USD','NQ=F']


In [20]:
conn = sqlite3.connect("market_data.db")
cursor = conn.cursor()

In [21]:
cursor.execute("DROP TABLE IF EXISTS prices")
conn.commit()

In [22]:
cursor.execute('''
    CREATE TABLE IF NOT EXISTS prices(
        ticker TEXT NOT NULL,
        date TEXT NOT NULL,
        open REAL,
        high REAL,
        low REAL,
        close REAL,
        volume REAL,
        PRIMARY KEY (ticker, date)
    )
''')

conn.commit()

In [26]:
for ticker in tickers:
    print(f"Fetching {ticker}...")
    data = yf.download(ticker, period='1y', interval='1d')

    #Flatenning API Fetched MultiIndex columns if present
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)


    for date, row in data.iterrows():
        cursor.execute("""
            INSERT OR IGNORE INTO prices (ticker, date, open, high, low, close, volume)
            VALUES (?,?,?,?,?,?,?)
        """, (
            ticker,
            date.strftime("%Y-%m-%d"),
            float(row["Open"]),
            float(row["High"]),
            float(row["Low"]),
            float(row["Close"]),
            float(row["Volume"])
        ))

    conn.commit()
    time.sleep(1)

print("Done.")

Fetching GC=F...


[*********************100%***********************]  1 of 1 completed


Fetching EURUSD=X...


[*********************100%***********************]  1 of 1 completed


Fetching GBPUSD=X...


[*********************100%***********************]  1 of 1 completed


Fetching BTC-USD...


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

Fetching NQ=F...


Done.


In [27]:
#Verifying
cursor.execute("SELECT ticker, COUNT(*) FROM prices GROUP BY ticker")
for row in cursor.fetchall():
    print(row)

('BTC-USD', 366)
('EURUSD=X', 260)
('GBPUSD=X', 260)
('GC=F', 252)
('NQ=F', 252)
